In [ ]:
# ==== THAM SỐ (cell duy nhất cần sửa) — đủ 8 CLI arg của src.evaluate (--run-dir = EXP_ROOT/<RUN_DIRS[i]>) ====
REPO_URL = "https://github.com/NguyenMinhTri24072005/MV_CAAFNet.git"
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/MV_CAAFNet"
DRIVE_DATA_SUBDIR = "data/processed"
LOCAL_DATA_DIR = "/content/processed"  # --data-dir
EXP_ROOT = f"{DRIVE_ROOT}/experiments"
VERIFY_H5_SHA256 = True

RUN_DIRS = [                           # danh sách đường dẫn tương đối theo EXP_ROOT, hoặc "ALL"
    "luna16/baseline_resunetpp_s42",
    "luna16/caaf_hausdorff_s42",
]
CKPTS = ["last"]                       # --ckpt (ví dụ ["last", "best"] cho S2)
SPLIT = "test"                         # --split ("test" | "val")
DATASET_OVERRIDE = None                # --dataset (None = đọc từ config_resolved.yaml)
BATCH_SIZE = None                      # --batch-size
THRESHOLD = None                       # --threshold
SAVE_PREDS = True                      # --save-preds / --no-save-preds

# 11 — Đánh giá (Phase 6–8b)

**Runtime:** Colab Pro **GPU**. Chỉ sửa **cell tham số** (đủ 8 CLI arg của `src.evaluate`). `RUN_DIRS = "ALL"` đánh giá
mọi run trong `EXP_ROOT` có checkpoint; `CKPTS = ["last", "best"]` cho S2. Kết quả ghi vào
`<run_dir>/eval_<split>_<ckpt>/` (`per_patch_metrics.csv`, `results.json`, `preds.npz`).

In [ ]:
# Setup chung: mount Drive → clone/pull repo (GH_TOKEN qua GIT_ASKPASS, không in token) → cài thư viện → in GPU
import json, os, platform, shlex, shutil, subprocess, sys
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = "/content/MV_CAAFNet"


def _git_env():
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")          # Colab Secrets → không ghi token vào notebook
    except Exception:
        token = None                               # repo public hoặc chưa cấu hình secret
    if token:
        askpass = Path("/content/.git_askpass.sh")
        askpass.write_text('#!/bin/sh\ncase "$1" in\n  Username*) echo "x-access-token" ;;\n  *) printf "%s" "$GH_TOKEN" ;;\nesac\n')
        askpass.chmod(0o700)
        env.update(GIT_ASKPASS=str(askpass), GH_TOKEN=token)
    return env


def _git(*args):
    subprocess.run(["git", *args], check=True, env=_git_env())


if Path(REPO_DIR, ".git").is_dir():
    _git("-C", REPO_DIR, "fetch", "--quiet", "origin", BRANCH)
    _git("-C", REPO_DIR, "checkout", "--quiet", BRANCH)
    _git("-C", REPO_DIR, "pull", "--quiet", "--ff-only", "origin", BRANCH)
else:
    _git("clone", "--quiet", "--branch", BRANCH, REPO_URL, REPO_DIR)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)


def run(args):
    """Chạy `python <args>` từ thư mục repo (danh sách tham số → an toàn với đường dẫn có dấu cách)."""
    cmd = [sys.executable, *map(str, args)]
    print("$", " ".join(shlex.quote(c) for c in cmd), flush=True)
    subprocess.run(cmd, check=True, cwd=REPO_DIR)


import torch

commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"commit {commit} | Python {platform.python_version()} | PyTorch {torch.__version__} | CUDA {torch.version.cuda}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}, {p.total_memory / 1024 ** 3:.1f} GB")
else:
    print("⚠ Không có GPU — Runtime → Change runtime type → GPU")

## 1. Danh sách run & dữ liệu cần dùng

In [ ]:
# Chép .h5 từ Drive về SSD /content (một lần mỗi phiên) và kiểm SHA-256 theo manifest.json
from src.data.pack_h5 import verify_manifest


def stage_dataset(ds, verify=True):
    src = Path(DRIVE_ROOT) / DRIVE_DATA_SUBDIR / ds
    dst = Path(LOCAL_DATA_DIR) / ds
    if not (src / "manifest.json").exists():
        raise FileNotFoundError(f"Chưa có {src}/manifest.json — chạy 01_colab_preprocess.ipynb trước")
    fresh = not ((dst / "manifest.json").exists()
                 and (dst / "manifest.json").read_bytes() == (src / "manifest.json").read_bytes())
    if fresh:
        shutil.rmtree(dst, ignore_errors=True)
        dst.mkdir(parents=True)
        for name in json.loads((src / "manifest.json").read_text(encoding="utf-8"))["files"]:
            shutil.copy2(src / name, dst / name)
        shutil.copy2(src / "manifest.json", dst / "manifest.json")  # chép cuối cùng
    if verify:
        bad = verify_manifest(dst)
        if bad:
            raise RuntimeError(f"SHA-256 sai ở {dst}: {bad} — xóa {dst} rồi chạy lại cell")
    print(f"{ds}: {'đã chép' if fresh else 'đã có sẵn'} → {dst}" + (" | SHA-256 OK" if verify else ""))

from src.utils import load_config

exp = Path(EXP_ROOT)


def resolve_runs(run_dirs):
    """`"ALL"` → mọi run trong EXP_ROOT có ít nhất một checkpoint thuộc CKPTS; ngược lại EXP_ROOT/<r>."""
    if run_dirs == "ALL":
        return sorted(p.parent for p in exp.glob("*/*/config_resolved.yaml")
                      if any((p.parent / f"{c}.pt").exists() for c in CKPTS))
    return [exp / r for r in run_dirs]


runs = resolve_runs(RUN_DIRS)
print(f"{len(runs)} run:", *[f"  {r.relative_to(exp)}" for r in runs], sep="\n")
for ds in sorted({DATASET_OVERRIDE or load_config(r / "config_resolved.yaml")["dataset"] for r in runs}):
    stage_dataset(ds, VERIFY_H5_SHA256)

## 2. Đánh giá

In [ ]:
for r in resolve_runs(RUN_DIRS):          # --run-dir = EXP_ROOT/<RUN_DIRS[i]>
    for ck in CKPTS:
        if not (r / f"{ck}.pt").exists():
            print(f"bỏ qua {r.name}: chưa có {ck}.pt")
            continue
        args = ["-m", "src.evaluate", "--run-dir", r, "--data-dir", LOCAL_DATA_DIR, "--ckpt", ck, "--split", SPLIT]
        if DATASET_OVERRIDE:
            args += ["--dataset", DATASET_OVERRIDE]
        if BATCH_SIZE:
            args += ["--batch-size", BATCH_SIZE]
        if THRESHOLD is not None:
            args += ["--threshold", THRESHOLD]
        args.append("--save-preds" if SAVE_PREDS else "--no-save-preds")
        run(args)

## 3. Tổng hợp (%, std ddof=1)

In [ ]:
import pandas as pd
from IPython.display import display

rows = []
for r in runs:
    for ck in CKPTS:
        f = r / f"eval_{SPLIT}_{ck}" / "results.json"
        if f.exists():
            res = json.loads(f.read_text(encoding="utf-8"))
            row = {"run": f"{r.parent.name}/{r.name}", "ckpt": ck, "epoch": res["epoch"], "n": res["n"]}
            for k, v in res["metrics"].items():
                row[k] = f"{v['mean']:.2f} ± {v['std']:.2f}" if v["mean"] is not None and v["std"] is not None else None
            row.update(w1=res["mean_w1"], w2=res["mean_w2"])
            rows.append(row)
display(pd.DataFrame(rows))